# Political Response Clarity — Classical NLP Baselines

This notebook compares TF-IDF and averaged Word2Vec features with class-balanced logistic regression on the QEvasion response-clarity task.

## Results status

The original Kaggle export did not preserve trustworthy executed outputs. Run the notebook end to end before reporting metrics; no score is claimed in this repository.

## Phase 0: Environment Setup

In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from datasets import load_dataset

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)

from gensim.models import KeyedVectors

from wordcloud import WordCloud

warnings.filterwarnings('ignore')

# Global random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Download NLTK data only if not already downloaded
def download_nltk_data():
    datasets = ['punkt', 'punkt_tab', 'stopwords', 'wordnet']
    for dataset in datasets:
        try:
            nltk.data.find(f'tokenizers/{dataset}' if 'punkt' in dataset else f'corpora/{dataset}')
            print(f'{dataset} already downloaded.')
        except LookupError:
            print(f'Downloading {dataset}...')
            nltk.download(dataset, quiet=True)

download_nltk_data()

print('Setup complete.')

## Phase 1: Load the Dataset

In [ ]:
# Load dataset with caching (HuggingFace datasets library caches automatically)
DATASET_REVISION = "3afc18f0b582b3cfdb927822cff57ddc6e871f9c"
dataset = load_dataset("ailsntua/QEvasion", revision=DATASET_REVISION, cache_dir="./data_cache")
print(dataset)
print(f"\nTrain size: {len(dataset['train'])}")
print(f"Test size: {len(dataset['test'])}")

In [ ]:
train_df = dataset['train'].to_pandas()
test_df = dataset['test'].to_pandas()

print('Column names:')
print(train_df.columns.tolist())
print(f'\nTrain shape: {train_df.shape}')
print(f'Test shape: {test_df.shape}')

In [ ]:
# Unique clarity labels
print('Clarity labels:', train_df['clarity_label'].unique())
print('\nLabel counts:')
print(train_df['clarity_label'].value_counts())

## Phase 2: Exploratory Data Analysis

### 2.1 Label Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Train set
label_counts_train = train_df['clarity_label'].value_counts()
label_counts_train.plot(kind='bar', ax=axes[0], color=['#2196F3', '#FF9800', '#4CAF50'], edgecolor='black')
axes[0].set_title('Train Set — Label Distribution')
axes[0].set_xlabel('Clarity Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(label_counts_train.values):
    axes[0].text(i, v + 20, str(v), ha='center', fontweight='bold')

# Test set
label_counts_test = test_df['clarity_label'].value_counts()
label_counts_test.plot(kind='bar', ax=axes[1], color=['#2196F3', '#FF9800', '#4CAF50'], edgecolor='black')
axes[1].set_title('Test Set — Label Distribution')
axes[1].set_xlabel('Clarity Label')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=15)
for i, v in enumerate(label_counts_test.values):
    axes[1].text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print('Train distribution (%):')
print((train_df['clarity_label'].value_counts(normalize=True) * 100).round(1))
print('\nObservation: The dataset is imbalanced — this will inform our use of class_weight="balanced".')

### 2.2 Text Length Analysis

In [ ]:
# Compute character and token counts
train_df['question_char_len'] = train_df['question'].astype(str).str.len()
train_df['answer_char_len'] = train_df['interview_answer'].astype(str).str.len()
train_df['question_token_len'] = train_df['question'].astype(str).str.split().str.len()
train_df['answer_token_len'] = train_df['interview_answer'].astype(str).str.split().str.len()

print('Text length statistics:')
print(train_df[['question_char_len', 'answer_char_len', 'question_token_len', 'answer_token_len']].describe().round(1))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

labels_order = train_df['clarity_label'].unique()
colors = {'Clear Reply': '#2196F3', 'Ambivalent': '#FF9800', 'Clear Non-Reply': '#4CAF50'}

for label in labels_order:
    subset = train_df[train_df['clarity_label'] == label]
    c = colors.get(label, 'gray')
    axes[0, 0].hist(subset['question_char_len'], bins=30, alpha=0.6, label=label, color=c)
    axes[0, 1].hist(subset['answer_char_len'], bins=30, alpha=0.6, label=label, color=c)
    axes[1, 0].hist(subset['question_token_len'], bins=30, alpha=0.6, label=label, color=c)
    axes[1, 1].hist(subset['answer_token_len'], bins=30, alpha=0.6, label=label, color=c)

axes[0, 0].set_title('Question — Character Length')
axes[0, 1].set_title('Answer — Character Length')
axes[1, 0].set_title('Question — Token Count')
axes[1, 1].set_title('Answer — Token Count')

for ax in axes.flat:
    ax.legend()
    ax.set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Mean answer length per class
print('Mean answer token length per class:')
print(train_df.groupby('clarity_label')['answer_token_len'].mean().round(1))
print('\nMean question token length per class:')
print(train_df.groupby('clarity_label')['question_token_len'].mean().round(1))

### 2.3 Vocabulary Analysis

In [ ]:
# Most frequent words per class
stop_words = set(stopwords.words('english'))

def get_top_words(texts, n=20):
    all_words = []
    for text in texts:
        tokens = str(text).lower().split()
        tokens = [t for t in tokens if t.isalpha() and t not in stop_words]
        all_words.extend(tokens)
    return Counter(all_words).most_common(n)

for label in train_df['clarity_label'].unique():
    subset = train_df[train_df['clarity_label'] == label]
    top = get_top_words(subset['interview_answer'])
    print(f'\nTop 15 words in answers for "{label}":')
    print(', '.join([f'{w}({c})' for w, c in top[:15]]))

In [ ]:
# Word clouds per class
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, label in enumerate(sorted(train_df['clarity_label'].unique())):
    subset = train_df[train_df['clarity_label'] == label]
    text = ' '.join(subset['interview_answer'].astype(str).tolist())
    wc = WordCloud(width=800, height=400, max_words=100,
                   background_color='white', stopwords=stop_words,
                   random_state=SEED).generate(text)
    axes[i].imshow(wc, interpolation='bilinear')
    axes[i].set_title(f'Word Cloud: {label}', fontsize=14)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

### 2.4 Missing / Edge Case Check

In [ ]:
print('Missing values in key columns (train):')
print(train_df[['question', 'interview_answer', 'clarity_label']].isnull().sum())

print('\nEmpty strings:')
print(f"  question: {(train_df['question'].astype(str).str.strip() == '').sum()}")
print(f"  interview_answer: {(train_df['interview_answer'].astype(str).str.strip() == '').sum()}")

print(f"\nVery short answers (< 10 chars): {(train_df['answer_char_len'] < 10).sum()}")
print(f"Multiple questions flag: {train_df['multiple_questions'].sum()} rows")

print('\nMissing values in key columns (test):')
print(test_df[['question', 'interview_answer', 'clarity_label']].isnull().sum())

## Phase 3: Text Preprocessing

**Steps applied:**
1. Lowercase — reduces vocabulary size without meaningful loss
2. Punctuation normalization — remove punctuation but preserve `?` (question marks carry signal)
3. Whitespace normalization — strip extra spaces/tabs/newlines
4. Tokenization — split into word tokens using NLTK
5. Stopword removal — applied manually (beneficial for both TF-IDF and Word2Vec)
6. Lemmatization — reduces inflected forms to their base for better generalization

In [ ]:
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    """Full preprocessing pipeline: lowercase, clean punctuation, tokenize, remove stopwords, lemmatize."""
    text = str(text).lower()
    # Remove punctuation except question marks
    text = re.sub(r'[^\w\s?]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t.isalpha() and t not in stop_words]
    return ' '.join(tokens)

# Apply preprocessing to both question and answer
train_df['question_clean'] = train_df['question'].apply(preprocess_text)
train_df['answer_clean'] = train_df['interview_answer'].apply(preprocess_text)
test_df['question_clean'] = test_df['question'].apply(preprocess_text)
test_df['answer_clean'] = test_df['interview_answer'].apply(preprocess_text)

print('Preprocessing complete.')
print('\nExample (original):')
print(train_df['interview_answer'].iloc[0][:200])
print('\nExample (preprocessed):')
print(train_df['answer_clean'].iloc[0][:200])

## Phase 4: Input Construction

Concatenate question and answer with a clear separator: `[Q]: <question> [A]: <answer>`

This allows the model to see the relationship between the question and the answer.

In [ ]:
# Construct combined input for both raw and preprocessed text
train_df['input_raw'] = '[Q]: ' + train_df['question'].astype(str) + ' [A]: ' + train_df['interview_answer'].astype(str)
test_df['input_raw'] = '[Q]: ' + test_df['question'].astype(str) + ' [A]: ' + test_df['interview_answer'].astype(str)

train_df['input_clean'] = '[Q]: ' + train_df['question_clean'] + ' [A]: ' + train_df['answer_clean']
test_df['input_clean'] = '[Q]: ' + test_df['question_clean'] + ' [A]: ' + test_df['answer_clean']

print('Input construction complete.')
print('\nExample combined input (clean):')
print(train_df['input_clean'].iloc[0][:300])

In [ ]:
# Encode labels
le = LabelEncoder()
y_train = le.fit_transform(train_df['clarity_label'])
y_test = le.transform(test_df['clarity_label'])

print('Label encoding:')
for i, label in enumerate(le.classes_):
    print(f'  {i} -> {label}')

## Phase 5A: TF-IDF Features + Logistic Regression

In [ ]:
# Build a pipeline: TfidfVectorizer -> LogisticRegression
tfidf_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(sublinear_tf=True)),
    ('clf', LogisticRegression(
        class_weight='balanced',
        solver='lbfgs',
        max_iter=1000,
        random_state=SEED
    ))
])

# Hyperparameter grid
tfidf_param_grid = {
    'tfidf__max_features': [10000, 30000, 50000],
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 5],
    'clf__C': [0.01, 0.1, 1, 10]
}

# Stratified 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

tfidf_grid = GridSearchCV(
    tfidf_pipeline,
    tfidf_param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Starting TF-IDF GridSearchCV...')
tfidf_grid.fit(train_df['input_clean'], y_train)

print(f'\nBest CV F1 (macro): {tfidf_grid.best_score_:.4f}')
print(f'Best parameters: {tfidf_grid.best_params_}')

In [ ]:
# Evaluate TF-IDF model on test set
y_pred_tfidf = tfidf_grid.predict(test_df['input_clean'])

print('=== TF-IDF + Logistic Regression — Test Set Results ===')
print(f'\nAccuracy: {accuracy_score(y_test, y_pred_tfidf):.4f}')
print(f'F1 (macro): {f1_score(y_test, y_pred_tfidf, average="macro"):.4f}')
print(f'Precision (macro): {precision_score(y_test, y_pred_tfidf, average="macro"):.4f}')
print(f'Recall (macro): {recall_score(y_test, y_pred_tfidf, average="macro"):.4f}')
print('\nPer-class report:')
print(classification_report(y_test, y_pred_tfidf, target_names=le.classes_))

## Phase 5B: Word2Vec Features + Logistic Regression

In [ ]:
# Load pre-trained Google News Word2Vec with local caching
import gensim.downloader as api

W2V_CACHE_DIR = './models_cache'
W2V_LOCAL_PATH = os.path.join(W2V_CACHE_DIR, 'GoogleNews-vectors-negative300.bin')
os.makedirs(W2V_CACHE_DIR, exist_ok=True)

if os.path.exists(W2V_LOCAL_PATH):
    print(f'Loading Word2Vec from local cache: {W2V_LOCAL_PATH}')
    w2v_model = KeyedVectors.load_word2vec_format(W2V_LOCAL_PATH, binary=True)
else:
    print('Downloading Word2Vec via gensim (cached for future runs)...')
    w2v_model = api.load('word2vec-google-news-300')
    print(f'Saving to local cache: {W2V_LOCAL_PATH}')
    w2v_model.save_word2vec_format(W2V_LOCAL_PATH, binary=True)

print(f'Word2Vec loaded. Vocabulary size: {len(w2v_model)}, Vector dimension: {w2v_model.vector_size}')

In [ ]:
def text_to_w2v_vector(text, model, dim=300):
    """Convert text to a fixed-size vector by averaging Word2Vec embeddings of its tokens."""
    tokens = str(text).split()
    vectors = []
    for token in tokens:
        if token in model:
            vectors.append(model[token])
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(dim)

print('Computing Word2Vec features for train set...')
X_train_w2v = np.array([text_to_w2v_vector(text, w2v_model) for text in train_df['input_clean']])

print('Computing Word2Vec features for test set...')
X_test_w2v = np.array([text_to_w2v_vector(text, w2v_model) for text in test_df['input_clean']])

print(f'Train W2V shape: {X_train_w2v.shape}')
print(f'Test W2V shape: {X_test_w2v.shape}')

# Check for zero vectors (all tokens OOV)
zero_train = np.sum(np.all(X_train_w2v == 0, axis=1))
zero_test = np.sum(np.all(X_test_w2v == 0, axis=1))
print(f'\nZero vectors (all OOV): train={zero_train}, test={zero_test}')

In [ ]:
# Word2Vec + Logistic Regression with GridSearchCV
w2v_param_grid = {
    'C': [0.01, 0.1, 1, 10],
}

w2v_grid = GridSearchCV(
    LogisticRegression(
        class_weight='balanced',
        solver='lbfgs',
        max_iter=1000,
        random_state=SEED
    ),
    w2v_param_grid,
    cv=cv,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
    refit=True
)

print('Starting Word2Vec GridSearchCV...')
w2v_grid.fit(X_train_w2v, y_train)

print(f'\nBest CV F1 (macro): {w2v_grid.best_score_:.4f}')
print(f'Best parameters: {w2v_grid.best_params_}')

In [ ]:
# Evaluate Word2Vec model on test set
y_pred_w2v = w2v_grid.predict(X_test_w2v)

print('=== Word2Vec + Logistic Regression — Test Set Results ===')
print(f'\nAccuracy: {accuracy_score(y_test, y_pred_w2v):.4f}')
print(f'F1 (macro): {f1_score(y_test, y_pred_w2v, average="macro"):.4f}')
print(f'Precision (macro): {precision_score(y_test, y_pred_w2v, average="macro"):.4f}')
print(f'Recall (macro): {recall_score(y_test, y_pred_w2v, average="macro"):.4f}')
print('\nPer-class report:')
print(classification_report(y_test, y_pred_w2v, target_names=le.classes_))

In [ ]:
# Comparison table
def compute_metrics(y_true, y_pred, model_name):
    return {
        'Model': model_name,
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1 (macro)': f1_score(y_true, y_pred, average='macro'),
        'Precision (macro)': precision_score(y_true, y_pred, average='macro'),
        'Recall (macro)': recall_score(y_true, y_pred, average='macro'),
    }

results = pd.DataFrame([
    compute_metrics(y_test, y_pred_tfidf, 'TF-IDF + LR'),
    compute_metrics(y_test, y_pred_w2v, 'Word2Vec + LR'),
])
results = results.set_index('Model')
print('=== Model Comparison ===')
print(results.round(4).to_string())

## Phase 6: Model Comparison

In [ ]:
# Per-class F1 comparison bar chart
f1_tfidf = f1_score(y_test, y_pred_tfidf, average=None)
f1_w2v = f1_score(y_test, y_pred_w2v, average=None)

x = np.arange(len(le.classes_))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, f1_tfidf, width, label='TF-IDF + LR', color='#2196F3', edgecolor='black')
bars2 = ax.bar(x + width/2, f1_w2v, width, label='Word2Vec + LR', color='#FF9800', edgecolor='black')

ax.set_xlabel('Class')
ax.set_ylabel('F1-Score')
ax.set_title('Per-Class F1-Score Comparison')
ax.set_xticks(x)
ax.set_xticklabels(le.classes_)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', fontsize=10)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.02,
            f'{bar.get_height():.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side confusion matrices
cm_tfidf = confusion_matrix(y_test, y_pred_tfidf)
cm_w2v = confusion_matrix(y_test, y_pred_w2v)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ConfusionMatrixDisplay(confusion_matrix=cm_tfidf, display_labels=le.classes_).plot(
    cmap='Blues', ax=axes[0], values_format='d')
axes[0].set_title('TF-IDF + LR')

ConfusionMatrixDisplay(confusion_matrix=cm_w2v, display_labels=le.classes_).plot(
    cmap='Oranges', ax=axes[1], values_format='d')
axes[1].set_title('Word2Vec + LR')

plt.suptitle('Confusion Matrix Comparison', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Phase 7: Error Analysis

In [ ]:
# Identify misclassified examples from the best model (TF-IDF)
test_df_analysis = test_df.copy()
test_df_analysis['pred_tfidf'] = le.inverse_transform(y_pred_tfidf)
test_df_analysis['pred_w2v'] = le.inverse_transform(y_pred_w2v)
test_df_analysis['true_label'] = le.inverse_transform(y_test)

# Misclassified by TF-IDF
misclassified_tfidf = test_df_analysis[test_df_analysis['pred_tfidf'] != test_df_analysis['true_label']]
print(f'TF-IDF misclassified: {len(misclassified_tfidf)} / {len(test_df_analysis)} ({100*len(misclassified_tfidf)/len(test_df_analysis):.1f}%)')
print(f'Word2Vec misclassified: {(test_df_analysis["pred_w2v"] != test_df_analysis["true_label"]).sum()} / {len(test_df_analysis)}')

print('\nMisclassification patterns (TF-IDF) — True vs Predicted:')
print(misclassified_tfidf.groupby(['true_label', 'pred_tfidf']).size().reset_index(name='count').to_string(index=False))

In [ ]:
# Error analysis: answer length vs correctness
test_df_analysis['answer_len'] = test_df_analysis['interview_answer'].astype(str).str.split().str.len()
test_df_analysis['tfidf_correct'] = test_df_analysis['pred_tfidf'] == test_df_analysis['true_label']

fig, ax = plt.subplots(figsize=(10, 5))
test_df_analysis.boxplot(column='answer_len', by='tfidf_correct', ax=ax)
ax.set_title('Answer Token Length: Correct vs Misclassified (TF-IDF)')
ax.set_xlabel('Correctly Classified')
ax.set_ylabel('Answer Token Length')
plt.suptitle('')
plt.tight_layout()
plt.show()

### Regularization Curves

In [ ]:
# Regularization curve: F1 vs regularization strength C for both models
C_values = [0.001, 0.01, 0.1, 1, 10, 100]

# Pre-compute TF-IDF features once (avoids re-vectorizing 30 times)
best_tfidf_params = tfidf_grid.best_params_
tfidf_vec = TfidfVectorizer(
    sublinear_tf=True,
    max_features=best_tfidf_params['tfidf__max_features'],
    ngram_range=best_tfidf_params['tfidf__ngram_range'],
    min_df=best_tfidf_params['tfidf__min_df']
)
X_train_tfidf = tfidf_vec.fit_transform(train_df['input_clean'])

# TF-IDF regularization curve (only vary LR's C)
tfidf_cv_scores = []
for c in C_values:
    lr = LogisticRegression(
        C=c, class_weight='balanced', solver='lbfgs',
        max_iter=1000, random_state=SEED
    )
    scores = cross_val_score(lr, X_train_tfidf, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    tfidf_cv_scores.append(scores.mean())

# Word2Vec regularization curve
w2v_cv_scores = []
for c in C_values:
    lr = LogisticRegression(
        C=c, class_weight='balanced', solver='lbfgs',
        max_iter=1000, random_state=SEED
    )
    scores = cross_val_score(lr, X_train_w2v, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    w2v_cv_scores.append(scores.mean())

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(C_values, tfidf_cv_scores, 'o-', label='TF-IDF + LR', color='#2196F3', linewidth=2)
ax.plot(C_values, w2v_cv_scores, 's-', label='Word2Vec + LR', color='#FF9800', linewidth=2)
ax.set_xscale('log')
ax.set_xlabel('Regularization Strength (C)')
ax.set_ylabel('Mean CV F1 (macro)')
ax.set_title('Regularization Curve: CV F1 vs C (Underfitting/Overfitting Check)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Learning Curves

In [ ]:
from sklearn.model_selection import learning_curve

train_sizes = np.linspace(0.1, 1.0, 8)

# --- TF-IDF learning curve (Pipeline ensures TF-IDF only sees training folds) ---
best_C_tfidf = tfidf_grid.best_params_['clf__C']
tfidf_lc_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(
        sublinear_tf=True,
        max_features=best_tfidf_params['tfidf__max_features'],
        ngram_range=best_tfidf_params['tfidf__ngram_range'],
        min_df=best_tfidf_params['tfidf__min_df']
    )),
    ('clf', LogisticRegression(
        C=best_C_tfidf, class_weight='balanced',
        solver='lbfgs', max_iter=1000, random_state=SEED
    ))
])
t_sizes_tfidf, t_scores_tfidf, v_scores_tfidf = learning_curve(
    tfidf_lc_pipe, train_df['input_clean'], y_train,
    train_sizes=train_sizes, cv=cv, scoring='f1_macro', n_jobs=-1
)

# --- Word2Vec learning curve ---
best_C_w2v = w2v_grid.best_params_['C']
w2v_lc = LogisticRegression(
    C=best_C_w2v, class_weight='balanced',
    solver='lbfgs', max_iter=1000, random_state=SEED
)
t_sizes_w2v, t_scores_w2v, v_scores_w2v = learning_curve(
    w2v_lc, X_train_w2v, y_train,
    train_sizes=train_sizes, cv=cv, scoring='f1_macro', n_jobs=-1
)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, title, sizes, t_sc, v_sc in [
    (axes[0], 'TF-IDF + LR',   t_sizes_tfidf, t_scores_tfidf, v_scores_tfidf),
    (axes[1], 'Word2Vec + LR', t_sizes_w2v,   t_scores_w2v,   v_scores_w2v),
]:
    t_mean, t_std = t_sc.mean(axis=1), t_sc.std(axis=1)
    v_mean, v_std = v_sc.mean(axis=1), v_sc.std(axis=1)

    ax.plot(sizes, t_mean, 'o-', color='#2196F3', label='Train score', linewidth=2)
    ax.fill_between(sizes, t_mean - t_std, t_mean + t_std, alpha=0.15, color='#2196F3')
    ax.plot(sizes, v_mean, 's-', color='#FF9800', label='CV score', linewidth=2)
    ax.fill_between(sizes, v_mean - v_std, v_mean + v_std, alpha=0.15, color='#FF9800')

    ax.set_title(f'Learning Curve — {title}')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('F1 (macro)')
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves: Train vs CV Score (Overfitting/Underfitting Check)', fontsize=14)
plt.tight_layout()
plt.show()

### Summary of Findings

**TF-IDF Strengths:**
- Captures task-specific vocabulary and keyphrases effectively
- Bigrams can capture short patterns like "I think", "you know" etc.
- Handles sparse, high-dimensional data well with Logistic Regression

**TF-IDF Weaknesses:**
- Cannot capture synonyms or semantic similarity
- Ignores word order beyond n-gram scope

**Word2Vec Strengths:**
- Captures semantic similarity between words
- Pre-trained on massive corpus provides rich representations

**Word2Vec Weaknesses:**
- Simple averaging loses word order and positional information
- Domain mismatch — general news corpus vs. political interviews
- Small corpus means averaging many OOV tokens

**Key observations:**
- The *Ambivalent* class is typically hardest to classify, being confused with both *Clear Reply* and *Clear Non-Reply*
- Class imbalance necessitates the use of `class_weight='balanced'` and evaluation via F1 rather than accuracy

## Phase 8: Export Predictions

In [ ]:
# Use the best model (TF-IDF, already retrained on full training set via refit=True)
best_model = tfidf_grid.best_estimator_
y_submission = best_model.predict(test_df['input_clean'])
y_submission_labels = le.inverse_transform(y_submission)

# Create submission DataFrame matching sample_solution.csv format
submission = pd.DataFrame({
    'Id': range(len(y_submission_labels)),
    'Predicted': y_submission_labels
})

submission.to_csv('submission.csv', index=False)

print('submission.csv generated successfully.')
print(f'Shape: {submission.shape}')
print(f'\nLabel distribution in submission:')
print(submission['Predicted'].value_counts())
print(f'\nAll rows:')
pd.set_option('display.max_rows', None)
submission